# Customer Segmentation with RFM Analysis
### Superstore Sales Dataset (2016–2019)

**Objective:** Segment customers using Recency, Frequency, and Monetary (RFM) analysis to identify high-value customers, at-risk customers, and opportunities for targeted marketing.


## 1. RFM Definitions & Business Context

### 1.1 What is RFM Analysis?

RFM is a behavioral customer segmentation technique that ranks and groups customers according to three transactional dimensions derived directly from purchase history:

| Metric | Definition | Business Question It Answers |
|---|---|---|
| **Recency (R)** | The number of days (or other time unit) elapsed since a customer's most recent purchase, measured against a fixed analysis/snapshot date. | *"How recently did this customer buy from us?"* |
| **Frequency (F)** | The total number of distinct purchase transactions (orders) a customer has made within the observation window. | *"How often does this customer buy from us?"* |
| **Monetary (M)** | The total monetary value (revenue/sales) a customer has generated across all their transactions within the observation window. | *"How much revenue does this customer generate?"* |

RFM rests on a well-established principle in marketing science and database marketing: **past transactional behavior is one of the strongest predictors of future purchasing behavior** — often more predictive than demographic or attitudinal data alone (Hughes, 1994; Fader, Hardie, & Lee, 2005).

### 1.2 Core Business Value of RFM Analysis

- **Prioritization of marketing spend:** Instead of treating all customers identically, RFM allows a firm to identify its most valuable customers (recent, frequent, high-spend) and its most at-risk or dormant customers, so limited marketing budget is directed where it will have the greatest return on investment.
- **Customer Lifetime Value (CLV) proxy:** RFM scores are a simple, low-cost, data-driven proxy for customer value and future purchase propensity, useful when a full probabilistic CLV model (e.g., BG/NBD or Pareto/NBD) is not yet feasible.
- **Actionable segmentation:** Because RFM is built entirely from transactional data already captured by any order-management or POS system, it requires no additional data collection and can be operationalized quickly into CRM campaigns (e.g., win-back emails for high-Recency-decline customers, loyalty rewards for high-Frequency customers).
- **Churn early-warning system:** A customer who was historically high-Frequency and high-Monetary but has a rapidly increasing Recency value is a strong candidate for proactive retention outreach before they fully churn.
- **Foundation for advanced analytics:** RFM scores are frequently used as engineered features in downstream machine learning models for churn prediction, next-purchase prediction, and customer lifetime value modeling.


## 2. RFM Calculation

The code below loads the Superstore transactional dataset and calculates the three RFM metrics **per `Customer ID`** using a single `.groupby().agg()` pipeline:

- **Recency** — derived from the `max()` of `Order Date` per customer (the customer's most recent order date), converted into "days since last purchase" relative to a fixed analysis/snapshot date.
- **Frequency** — the `nunique()` count of `Order ID` per customer (distinct orders placed).
- **Monetary** — the `sum()` of `Sales` per customer (total revenue generated).

The resulting table is ranked with `.sort_values()` by Monetary value, from highest to lowest.


In [1]:
import pandas as pd

# Load the transactional data
df = pd.read_csv('Sample-Superstore2019.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Define the analysis (snapshot) date
analysis_date = df['Order Date'].max() + pd.Timedelta(days=1)

# Aggregate RFM building blocks per customer
rfm = df.groupby('Customer ID').agg(
    Recency = ('Order Date', lambda x: (analysis_date - x.max()).days),
    Frequency = ('Order ID', 'nunique'),
    Monetary = ('Sales', 'sum')
)

# --- Quantile (Quintile) Scoring: scale of 1 (worst) to 5 (best) ---
# Recency is scored INVERSELY: a LOWER number of days since last purchase
# is more desirable, so it receives a HIGHER score.
r_labels = range(5, 0, -1)   # 5 = most recent, 1 = least recent
f_labels = range(1, 6)       # 5 = most frequent, 1 = least frequent
m_labels = range(1, 6)       # 5 = highest spend,  1 = lowest spend

# .rank(method='first') breaks ties before qcut, guaranteeing five
# equal-sized quintile bins even when many customers share the same
# Frequency (a common issue with discrete, low-cardinality counts).
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), q=5, labels=r_labels).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=f_labels).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=m_labels).astype(int)

# Combine the three individual scores into a single RFM_Score string (e.g., '555')
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
)

# Also compute a summed composite score (3-15) for straightforward overall ranking
rfm['RFM_Total_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

# Sort scientifically: highest overall RFM score first, using Monetary value
# as a tie-breaker among customers who share the same total score
rfm = rfm.sort_values(by=['RFM_Total_Score', 'Monetary'], ascending=[False, False])

rfm.head(10)

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total_Score
Customer ID,,,,,,,,
SE-20110,10,11,12209.4380,5,5,5,555,15
JL-15835,22,11,9799.9230,5,5,5,555,15
PK-19075,10,12,8646.9340,5,5,5,555,15
HM-14860,3,10,8236.7648,5,5,5,555,15
LC-16885,17,12,7663.1260,5,5,5,555,15
PO-18850,5,11,7473.8282,5,5,5,555,15
DR-12880,4,9,6528.0340,5,5,5,555,15
JG-15160,2,11,6366.3920,5,5,5,555,15
WB-21850,21,11,6160.1020,5,5,5,555,15


## 3. Scientific Method for Customer Ranking & Scoring

### 3.1 Quartile (Quantile) Scoring Methodology

The most widely adopted and academically supported approach for converting raw RFM values into comparable scores is **quantile-based scoring**, typically using quartiles (1–4) or quintiles (1–5). The logic:

1. **Rank customers independently on each of the three dimensions** (R, F, and M) using their empirical distribution — not fixed thresholds — since transactional data is almost always heavily right-skewed rather than normally distributed.
2. **Split each dimension into equal-sized bins (quantiles)** using a method such as `pandas.qcut()`, which bins customers by rank rather than by raw value range. This ensures each score bucket contains an equal (or near-equal) proportion of the customer base, which is the standard practice for handling skewed marketing data.
3. **Assign a score of 1–4 (quartiles) or 1–5 (quintiles) to each dimension**, where:
   - For **Frequency** and **Monetary**, a *higher* raw value receives a *higher* score (4 = highest frequency/spend quartile).
   - For **Recency**, the scoring is *inverted*: a *lower* number of days since last purchase receives a *higher* score (4 = most recently active quartile), because a small Recency value represents more desirable (recent) behavior.
4. **Concatenate the three individual scores into a combined RFM score** (e.g., R=4, F=3, M=4 → `"434"`), or sum them into a single composite score, depending on whether the analysis calls for granular segmentation or a single ranking index.
5. **Map combined scores to named business segments** (e.g., "Champions," "Loyal Customers," "At Risk," "Hibernating," "Lost") using a segmentation matrix — a widely used reference implementation is the RFM segment grid popularized by Putler/Fader, which cross-tabulates the R score against the combined F+M score.

### 3.2 Alternative / Complementary Method: Pareto (80/20) Analysis

As a complementary sense-check, the **Pareto principle** can be applied to the Monetary dimension: rank customers by cumulative Monetary contribution and identify the approximate proportion of customers generating ~80% of total revenue. This is a useful executive-level summary statistic alongside the more granular quartile segmentation, and is well documented in marketing and retail analytics literature as a heuristic for concentration of customer value.

### 3.3 Why Quantile Scoring Is Scientifically Preferred Over Fixed Thresholds

- It is **distribution-agnostic** — it does not assume normality, which is important because Recency, Frequency, and Monetary in retail data are almost universally right-skewed.
- It produces **balanced segment sizes**, which is operationally important for designing campaigns and allocating marketing resources proportionally.
- It is **reproducible and scalable** across different customer bases, time periods, or business units without manually re-deriving thresholds each time.

### 3.4 References

1. Hughes, A. M. (1994). *Strategic Database Marketing: The Masterplan for Starting and Managing a Profitable, Customer-Based Marketing Program.* Chicago: Probus Publishing. — Foundational text credited with popularizing the RFM framework in direct and database marketing.
2. Fader, P. S., Hardie, B. G. S., & Lee, K. L. (2005). "RFM and CLV: Using Iso-Value Curves for Customer Base Analysis." *Journal of Marketing Research*, 42(4), 415–430. — Connects RFM scoring to formal Customer Lifetime Value modeling.
3. McCarty, J. A., & Hastak, M. (2007). "Segmentation Approaches in Data-Mining: A Comparison of RFM, CHAID, and Logistic Regression." *Journal of Business Research*, 60(6), 656–662. — Empirical comparison validating RFM's effectiveness relative to other segmentation techniques.
4. Cheng, C.-H., & Chen, Y.-S. (2009). "Classifying the Segmentation of Customer Value via RFM Model and RS Theory." *Expert Systems with Applications*, 36(3), 4176–4184. — Academic treatment of quantile/quartile-based RFM scoring and segmentation.
5. Chen, D., Sain, S. L., & Guo, K. (2012). "Data Mining for the Online Retail Industry: A Case Study of RFM Model-Based Customer Segmentation Using Data Mining." *Journal of Database Marketing & Customer Strategy Management*, 19(3), 197–208. — Applied retail case study using quartile RFM scoring, methodologically similar to this notebook's dataset.
6. Wei, J.-T., Lin, S.-Y., & Wu, H.-H. (2010). "A Review of the Application of RFM Model." *African Journal of Business Management*, 4(19), 4199–4206. — Literature review consolidating RFM scoring variants (quartile, quintile, weighted RFM) across industries.


## 4. Recommended Visualizations for RFM Results

*(No chart code is generated here per assignment scope — this section describes recommended visualizations and their business rationale.)*

### 4.1 Treemap of Customer Segments
**What it shows:** Each named RFM segment (e.g., Champions, Loyal, At Risk, Lost) as a rectangle, sized by number of customers or total Monetary contribution, and often color-coded by average RFM score.

**Why it's useful:** A treemap communicates both the *relative size* and *relative value* of every segment in a single glance — decision-makers can immediately see, for example, that "Champions" is a small rectangle by customer count but a large rectangle by revenue share, which is a compelling visual argument for prioritizing retention spend on that group.

### 4.2 RFM Heatmap (Recency vs. Frequency, colored by average Monetary value or customer count)
**What it shows:** A grid with Recency score on one axis and Frequency score on the other, where each cell's color intensity represents either the count of customers in that R–F combination or their average Monetary value.

**Why it's useful:** Heatmaps make it easy to visually locate concentration "hot spots" — e.g., a cluster of customers with poor Recency but historically high Frequency signals a high-value, high-risk group that warrants an urgent win-back campaign. This is one of the most standard visualizations in RFM reporting because it directly supports the two-dimensional segmentation grids used to define named customer segments.

### 4.3 Bar Chart of Customer Count / Revenue by Segment
**What it shows:** A simple horizontal or vertical bar chart with named segments (Champions, Loyal Customers, At Risk, Hibernating, Lost, etc.) on one axis and either customer count or total revenue on the other.

**Why it's useful:** Bar charts are the clearest way to communicate segment prioritization to non-technical stakeholders (e.g., marketing or executive teams) — they answer "how many customers are in each bucket, and how much are they worth?" without requiring the audience to interpret a multi-dimensional plot.

### 4.4 Scatter Plot of Frequency vs. Monetary, colored by Recency score
**What it shows:** Each customer plotted as a point with Frequency on the x-axis and Monetary value on the y-axis, colored (or sized) by their Recency score.

**Why it's useful:** This exposes the underlying relationship between purchase frequency and spend while layering in recency as a third dimension, helping analysts spot patterns such as high-frequency/high-spend customers who are nonetheless trending toward inactivity (a valuable insight a table of numbers alone would not surface as quickly).

### 4.5 Distribution Histograms of Raw R, F, and M values (pre-scoring)
**What it shows:** Three histograms, one per RFM dimension, showing the raw (unscored) distribution of values across the customer base.

**Why it's useful:** Before committing to a quartile/quintile cut, it's important to visually confirm the skewness of each metric — this justifies the choice of quantile-based scoring (Section 3) over naive equal-width binning, and is a standard diagnostic step in any rigorous RFM methodology write-up.
